Run this notebook in a clean Kaggle session with a Tesla P100 GPU and the repository plus AI4Mars dataset attached. Run the dependency cell before any torch import; do not install `requirements.txt`. The CUDA preflight must pass and the smoke configuration should complete before starting the full run.

In [ ]:
import subprocess
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'pyproject.toml').is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(PROJECT_ROOT / 'requirements-kaggle.txt')])

In [ ]:
import os
from pathlib import Path

from ai4mars import run_full_reproduction

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'pyproject.toml').is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent

configured_dataset = os.environ.get('AI4MARS_DATASET_ROOT')
if configured_dataset:
    DATASET_ROOT = Path(configured_dataset)
else:
    matches = list(Path('/kaggle/input').glob('**/ai4mars-dataset-merged-0.6'))
    if len(matches) != 1:
        raise RuntimeError('Set AI4MARS_DATASET_ROOT to the extracted AI4Mars merged 0.6 directory.')
    DATASET_ROOT = matches[0]

CONFIG_PATH = PROJECT_ROOT / 'configs' / 'reproduction' / 'paper_deeplabv3plus_kaggle_p100.yaml'
MANIFEST_ROOT = PROJECT_ROOT / 'artifacts' / 'manifests'
OUTPUT_ROOT = Path('/kaggle/working/ai4mars-paper-reproduction')
RESUME_CHECKPOINT = Path(os.environ['AI4MARS_RESUME_CHECKPOINT']) if os.environ.get('AI4MARS_RESUME_CHECKPOINT') else None

In [ ]:
run_dir = run_full_reproduction(
    config_path=CONFIG_PATH,
    dataset_root=DATASET_ROOT,
    manifest_root=MANIFEST_ROOT,
    output_root=OUTPUT_ROOT,
    resume_checkpoint=RESUME_CHECKPOINT,
    device='cuda',
)
print(f'Completed run: {run_dir}')

In [ ]:
import json

import matplotlib.pyplot as plt

epochs = [json.loads(line) for line in (run_dir / 'metrics.jsonl').read_text(encoding='utf-8').splitlines()]
epoch_numbers = [item['epoch'] for item in epochs]
figure, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epoch_numbers, [item['train_loss'] for item in epochs], label='Train')
axes[0].plot(epoch_numbers, [item['val_loss'] for item in epochs], label='Validation')
axes[0].set(title='Loss by completed epoch', xlabel='Epoch', ylabel='Cross-entropy')
axes[0].legend()
axes[1].plot(epoch_numbers, [item['mean_iou'] for item in epochs], color='#047857')
axes[1].set(title='Validation mIoU by completed epoch', xlabel='Epoch', ylabel='mIoU', ylim=(0, 1))
figure.tight_layout()
plt.show()